# Projeto de Visualização Computacional (SCC5836)
## Etapa 1: Pré-processamento, Limpeza e Engenharia de Features
**ICMC - Universidade de São Paulo (USP)**  

**Professor:** Jean R. Ponciano  

**Grupo:**  
- Camila Donda Ronchi  
- Henrique Gualberto Marques  
- Sandra Bastos de Almeida  

Este notebook processa o conjunto de dados bruto extraído do sistema Janus (`disciplinas.json`), trata inconsistências/valores nulos, valida as regras de negócio de créditos e cargas horárias da USP e define features para as análises visuais subsequentes.

> **Compatibilidade:** Este notebook está preparado para rodar tanto **localmente** (em sua IDE de preferência) quanto no **Google Colab**. Foram realizados testes em ambos os ambientes.


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Configurações de exibição do Pandas
pd.set_option('display.max_columns', 35)
pd.set_option('display.width', 1000)

# Constante para identificar ambiente (Google Colab ou Local)
IN_COLAB = 'google.colab' in sys.modules

In [ ]:
# Lista de caminhos possíveis para encontrar o arquivo "disciplinas.json"
path_list = [
    Path('../data/disciplinas.json'),
    Path('data/disciplinas.json'),
    Path('disciplinas.json'),
    Path('/content/data/disciplinas.json'),
    Path('/content/disciplinas.json')
]

data_path = None
for path in path_list:
    if path.exists():
        data_path = path
        break

# Validando se o arquivo de input existe
if data_path is None:
    if IN_COLAB:
        raise FileNotFoundError(
            'Arquivo disciplinas.json não encontrado no Google Colab! '
            'Faça o upload do arquivo disciplinas.json arrastando-o para a aba "Arquivos" do Colab.'
        )
    else:
        raise FileNotFoundError('Arquivo disciplinas.json não encontrado. Verifique a pasta data/.')

output_dir = data_path.parent
if str(output_dir) == '.' or str(output_dir) == '':
    output_dir = Path('.')

print(f'Ambiente detectado: {"Google Colab" if IN_COLAB else "Local"}')
print(f'Arquivo de input localizado em: {data_path.resolve()}')
print(f'Diretório de output para os arquivos limpos: {output_dir.resolve()}')


### 1. Ingestão dos Dados Brutos do Janus

In [ ]:
# Carregamento do arquivo "disciplinas.json" gerado pelo crawler Scrapy.
df_raw = pd.read_json(data_path)
print(f'Dimensão original dos dados brutos: {df_raw.shape[0]} linhas e {df_raw.shape[1]} colunas.')
df_raw.head(5)

### 2. Tratamento de Dados Nulos e Incompletos

In [ ]:
# Identificando disciplinas sem créditos/carga horária cadastrados no Janus (descontinuadas ou sem turma ativa)
na_data = df_raw['n_creditos'].isna()
df_incomplete = df_raw[na_data].copy()
df = df_raw[~na_data].copy()

loss_perc = (len(df_incomplete) / len(df_raw)) * 100
print(f'Registros com ementa incompleta no Janus: {len(df_incomplete)} ({loss_perc:.2f}%)')
print(f'Registros válidos mantidos para a análise: {len(df)} ({100 - loss_perc:.2f}%)')

### 3. Tipagem dos Dados

In [ ]:
# Definindo tipagem dos dados numéricos
df['n_creditos'] = df['n_creditos'].astype(int)
df['carga_teorica'] = df['carga_teorica'].astype(int)
df['carga_pratica'] = df['carga_pratica'].astype(int)
df['carga_estudo'] = df['carga_estudo'].astype(int)

# Tratamento das colunas textuais de tempo (duracao (ex: "15 semanas") e carga_total (ex: "120 horas"))
df['duracao_semanas'] = df['duracao'].astype(str).str.extract(r'(\d+)').astype(int)
df['carga_total_horas'] = df['carga_total'].astype(str).str.extract(r'(\d+)').astype(int)
df['ano_criacao'] = pd.to_datetime(df['criacao'], format='%d/%m/%Y', errors='coerce').dt.year

In [ ]:
# Ajuste de grafia de "comissão"
df['comissao'] = df['commissao']
df['codigo_comissao'] = df['codigo_commissao']

In [ ]:
print('Tipagem dos dados concluída com sucesso!')
df.head(5)

### 4. Validação Matemática das Regras Institucionais da USP
Validação das seguintes regras:
1. `Carga Total (horas) = (Teórica + Prática + Estudo) * Semanas`
2. `Horas por Crédito = Carga Total (horas) / Nº de Créditos == 15.0`

In [ ]:
horas_calc = (df['carga_teorica'] + df['carga_pratica'] + df['carga_estudo']) * df['duracao_semanas']
val_horas = (horas_calc == df['carga_total_horas']).all()
val_creditos = (df['carga_total_horas'] / df['n_creditos'] == 15.0).all()

print(f'Equivalência Carga Semanal * Duração == Carga Total: {val_horas} (100% dos dados)')
print(f'Equivalência Carga Total / Créditos == 15.0: {val_creditos} (100% dos dados)')

### 5. Engenharia de Features e Categorizações Institucionais/Geográficas

In [ ]:
# Somando cargas horárias (aula e total)
df['carga_aula_semanal'] = df['carga_teorica'] + df['carga_pratica']
df['carga_semanal_total'] = df['carga_aula_semanal'] + df['carga_estudo']

In [ ]:
# Proporções relativas ao tipo de carga horária (prática, teórica, estudo)
df['prop_pratica'] = np.where(df['carga_aula_semanal'] > 0, (df['carga_pratica'] / df['carga_aula_semanal']).round(3), 0.0)
df['prop_teorica'] = np.where(df['carga_aula_semanal'] > 0, (df['carga_teorica'] / df['carga_aula_semanal']).round(3), 0.0)
df['prop_estudo'] = np.where(df['carga_semanal_total'] > 0, (df['carga_estudo'] / df['carga_semanal_total']).round(3), 0.0)
df['razao_estudo_aula'] = np.where(df['carga_aula_semanal'] > 0, (df['carga_estudo'] / df['carga_aula_semanal']).round(2), np.nan)

In [ ]:
# Formato e duração da disciplina (disciplinas intensivas possuem 4 semanas ou menos)
df['is_intensiva'] = df['duracao_semanas'] <= 4
df['tipo_duracao'] = np.where(df['is_intensiva'], 'Intensiva (<= 4 sem)', 'Regular (> 4 sem)')

In [ ]:
# Categorização de idiomas
def categorizar_idioma(x):
    s = str(x).strip()
    if 'Português' in s and 'Inglês' in s: return 'Bilíngue (Português/Inglês)'
    elif 'Inglês' in s: return 'Inglês'
    elif 'Português' in s: return 'Português'
    elif 'Espanhol' in s: return 'Espanhol'
    elif not s or s.lower() in ['none', 'nan', '']: return 'Não especificado'
    else: return f'Outro ({s})'

df['idioma_cat'] = df['idioma'].apply(categorizar_idioma)

# Padronização de campos vazios de oferecimento
df['oferecimento_cat'] = df['oferecimento'].replace('', 'Não especificado')

In [ ]:
# Contagem de docentes
df['num_docentes'] = df['docentes_responsaveis'].apply(
    lambda x: len([d for d in str(x).split(' | ') if d.strip()]) if isinstance(x, str) and x.strip() else 0
)

In [ ]:
# Classificação geográfica (campus da USP) e acadêmica (grande área)

# Dicionário explícito das comissões de pós-graduação do Janus
MAPEAMENTO_CPG = {
    "Bioenergia da Usp, Unesp e Unicamp": ("Intercampi / Interinstitucional", "Multidisciplinar"),
    "Centro de Energia Nuclear na Agricultura": ("Piracicaba", "Ciências Agrárias"),
    "Escola Politécnica": ("São Paulo", "Engenharias"),
    'Escola Superior de Agricultura "Luiz de Queiroz"': ("Piracicaba", "Ciências Agrárias"),
    "Escola de Artes, Ciências e Humanidades": ("São Paulo", "Humanidades e Sociais Aplicadas"),
    "Escola de Comunicações e Artes": ("São Paulo", "Humanidades e Sociais Aplicadas"),
    "Escola de Educação Física e Esporte": ("São Paulo", "Ciências Biológicas e da Saúde"),
    "Escola de Educação Física e Esporte de Ribeirão Preto": ("Ribeirão Preto", "Ciências Biológicas e da Saúde"),
    "Escola de Enfermagem": ("São Paulo", "Ciências Biológicas e da Saúde"),
    "Escola de Enfermagem de Ribeirão Preto": ("Ribeirão Preto", "Ciências Biológicas e da Saúde"),
    "Escola de Engenharia de Lorena": ("Lorena", "Engenharias"),
    "Escola de Engenharia de São Carlos": ("São Carlos", "Engenharias"),
    "Faculdade de Arquitetura e Urbanismo e de Design": ("São Paulo", "Humanidades e Sociais Aplicadas"),
    "Faculdade de Ciências Farmacêuticas": ("São Paulo", "Ciências Biológicas e da Saúde"),
    "Faculdade de Ciências Farmacêuticas de Ribeirão Preto": ("Ribeirão Preto", "Ciências Biológicas e da Saúde"),
    "Faculdade de Direito": ("São Paulo", "Humanidades e Sociais Aplicadas"),
    "Faculdade de Direito de Ribeirão Preto": ("Ribeirão Preto", "Humanidades e Sociais Aplicadas"),
    "Faculdade de Economia, Administração e Contabilidade de Ribeirão Preto": ("Ribeirão Preto", "Humanidades e Sociais Aplicadas"),
    "Faculdade de Economia, Administração, Contabilidade e Atuária": ("São Paulo", "Humanidades e Sociais Aplicadas"),
    "Faculdade de Educação": ("São Paulo", "Humanidades e Sociais Aplicadas"),
    "Faculdade de Filosofia, Ciências e Letras de Ribeirão Preto": ("Ribeirão Preto", "Multidisciplinar"),
    "Faculdade de Filosofia, Letras e Ciências Humanas": ("São Paulo", "Humanidades e Sociais Aplicadas"),
    "Faculdade de Medicina": ("São Paulo", "Ciências Biológicas e da Saúde"),
    "Faculdade de Medicina Veterinária e Zootecnia": ("São Paulo", "Ciências Agrárias"),
    "Faculdade de Medicina de Bauru": ("Bauru", "Ciências Biológicas e da Saúde"),
    "Faculdade de Medicina de Ribeirão Preto": ("Ribeirão Preto", "Ciências Biológicas e da Saúde"),
    "Faculdade de Odontologia": ("São Paulo", "Ciências Biológicas e da Saúde"),
    "Faculdade de Odontologia de Bauru": ("Bauru", "Ciências Biológicas e da Saúde"),
    "Faculdade de Odontologia de Ribeirão Preto": ("Ribeirão Preto", "Ciências Biológicas e da Saúde"),
    "Faculdade de Odontologia, Escola de Enfermagem e Faculdade de Saúde Pública": ("Intercampi / Interinstitucional", "Ciências Biológicas e da Saúde"),
    "Faculdade de Saúde Pública": ("São Paulo", "Ciências Biológicas e da Saúde"),
    "Faculdade de Zootecnia e Engenharia de Alimentos": ("Pirassununga", "Ciências Agrárias"),
    "Instituto Dante Pazzanese de Cardiologia": ("São Paulo", "Ciências Biológicas e da Saúde"),
    "Instituto Oceanográfico": ("São Paulo", "Ciências Exatas e da Terra"),
    "Instituto de Arquitetura e Urbanismo": ("São Carlos", "Humanidades e Sociais Aplicadas"),
    "Instituto de Astronomia, Geofísica e Ciências Atmosféricas": ("São Paulo", "Ciências Exatas e da Terra"),
    "Instituto de Biociências": ("São Paulo", "Ciências Biológicas e da Saúde"),
    "Instituto de Ciências Biomédicas": ("São Paulo", "Ciências Biológicas e da Saúde"),
    "Instituto de Ciências Matemáticas e de Computação": ("São Carlos", "Ciências Exatas e da Terra"),
    "Instituto de Energia e Ambiente": ("São Paulo", "Multidisciplinar"),
    "Instituto de Estudos Brasileiros": ("São Paulo", "Humanidades e Sociais Aplicadas"),
    "Instituto de Física": ("São Paulo", "Ciências Exatas e da Terra"),
    "Instituto de Física de São Carlos": ("São Carlos", "Ciências Exatas e da Terra"),
    "Instituto de Geociências": ("São Paulo", "Ciências Exatas e da Terra"),
    "Instituto de Matemática, Estatística e Ciência da Computação": ("São Paulo", "Ciências Exatas e da Terra"),
    "Instituto de Pesquisas Energéticas e Nucleares": ("São Paulo", "Engenharias"),
    "Instituto de Psicologia": ("São Paulo", "Humanidades e Sociais Aplicadas"),
    "Instituto de Química": ("São Paulo", "Ciências Exatas e da Terra"),
    "Instituto de Química de São Carlos": ("São Carlos", "Ciências Exatas e da Terra"),
    "Instituto de Relações Internacionais": ("São Paulo", "Humanidades e Sociais Aplicadas"),
    "Interinstitucional em Estatística do Icmc e Ufscar": ("Intercampi / Interinstitucional", "Ciências Exatas e da Terra"),
    "Interunidades Sistemas Integrados em Alimentos": ("Intercampi / Interinstitucional", "Multidisciplinar"),
    "Interunidades em Bioengenharia": ("Intercampi / Interinstitucional", "Engenharias"),
    "Interunidades em Bioinformática": ("Intercampi / Interinstitucional", "Multidisciplinar"),
    "Interunidades em Biotecnologia": ("Intercampi / Interinstitucional", "Multidisciplinar"),
    "Interunidades em Ecologia Aplicada": ("Intercampi / Interinstitucional", "Ciências Agrárias"),
    "Interunidades em Enfermagem": ("Intercampi / Interinstitucional", "Ciências Biológicas e da Saúde"),
    "Interunidades em Ensino de Ciências": ("Intercampi / Interinstitucional", "Multidisciplinar"),
    "Interunidades em Estética e História da Arte": ("Intercampi / Interinstitucional", "Humanidades e Sociais Aplicadas"),
    "Interunidades em Integração da América Latina": ("Intercampi / Interinstitucional", "Humanidades e Sociais Aplicadas"),
    "Interunidades em Museologia": ("Intercampi / Interinstitucional", "Humanidades e Sociais Aplicadas"),
    "Museu de Arqueologia e Etnologia": ("São Paulo", "Humanidades e Sociais Aplicadas"),
    "Museu de Zoologia": ("São Paulo", "Ciências Biológicas e da Saúde"),
    "Pró-Reitoria de Pós-Graduação": ("Intercampi / Interinstitucional", "Multidisciplinar"),
}

In [ ]:
# Aplicando o mapeamento para os campos novos
df["campus"] = df["comissao"].map(
    {comissao: classificacao[0] for comissao, classificacao in MAPEAMENTO_CPG.items()}
)

df["grande_area"] = df["comissao"].map(
    {comissao: classificacao[1] for comissao, classificacao in MAPEAMENTO_CPG.items()}
)

In [ ]:
# Visualizando o mapeamento gerado
print(f"CPGs mapeadas: {len(MAPEAMENTO_CPG)}")
print(f"Campus distintos: {df['campus'].nunique()}")
print(f"Grandes áreas distintas: {df['grande_area'].nunique()}")

display(
    df[["comissao", "campus", "grande_area"]]
    .drop_duplicates()
    .sort_values(["campus", "grande_area", "comissao"])
    .reset_index(drop=True)
)

In [ ]:
# Sumário descritivo das features numéricas
display_cols = ['n_creditos', 'duracao_semanas', 'carga_total_horas', 'carga_teorica', 'carga_pratica', 'carga_estudo', 'prop_pratica', 'prop_estudo']
display(df[display_cols].describe().round(2))

In [ ]:
# Sumário da distribuição por campus
display(df['campus'].value_counts())

In [ ]:
# Sumário da distribuição por grande área
print('\nDISTRIBUIÇÃO POR GRANDE ÁREA')
display(df['grande_area'].value_counts())

### 6. Exportação dos Dados Tratados
Exportação em `.csv` e `.json` para consumo direto na etapa de visualização.

In [ ]:
columns_list = [
    'codigo', 'disciplina', 'grande_area', 'campus', 'comissao', 'codigo_comissao',
    'nome_programa', 'codigo_programa', 'area_concentracao', 'codigo_area_concentracao',
    'n_creditos', 'carga_total_horas', 'duracao_semanas', 'tipo_duracao', 'is_intensiva',
    'carga_teorica', 'carga_pratica', 'carga_estudo', 'carga_aula_semanal', 'carga_semanal_total',
    'prop_teorica', 'prop_pratica', 'prop_estudo', 'razao_estudo_aula',
    'idioma_cat', 'oferecimento_cat', 'num_docentes', 'docentes_responsaveis', 'ano_criacao',
    'objetivos', 'justificativa', 'conteudo', 'avaliacao', 'bibliografia'
]

df_clean = df[columns_list].copy()

path_csv = output_dir / 'disciplinas_limpo.csv'
path_json = output_dir / 'disciplinas_limpo.json'

df_clean.to_csv(path_csv, index=False, encoding='utf-8')
df_clean.to_json(path_json, orient='records', force_ascii=False, indent=2)

print('Base limpa exportada com sucesso:')
print(f'- CSV: {path_csv} ({path_csv.stat().st_size / (1024*1024):.2f} MB)')
print(f'- JSON: {path_json} ({path_json.stat().st_size / (1024*1024):.2f} MB)')
